#### Generate fake data

In [ ]:
# add faker to Packages
from faker import Faker
from random import randrange

f = Faker()
output = [[f.name(), f.address(), f.city(), f.state(), f.email(), 10 + randrange(70)]
    for _ in range(1000)]
output

#### Connect to Snowflake

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.get_current_warehouse(), session.get_current_role()

#### Create Snowpark DataFrame

In [ ]:
from snowflake.snowpark.types import StructType, StructField, StringType, IntegerType

schema = StructType([ 
    StructField("NAME", StringType(), False),  
    StructField("ADDRESS", StringType(), False), 
    StructField("CITY", StringType(), False),  
    StructField("STATE", StringType(), False),  
    StructField("EMAIL", StringType(), False),
    StructField("AGE", IntegerType(), False)
])
df = session.create_dataframe(output, schema=schema)
df

#### Save in table

In [ ]:
df.write.mode("overwrite").save_as_table("customers_fake")

#### Transform data

In [ ]:
df = session.table("customers_fake")
df.update({"AGE": 20}, df["AGE"] < 20)

In [ ]:
update customers_fake
set age = 20
where age < 20

#### Check back data

In [ ]:
select *
from customers_fake
limit 1000

In [ ]:
query = 'select * from customers_fake limit 1000'
df = session.sql(query).collect()
df

#### Show a histogram with Matplotlib

In [ ]:
# add matplotlib to Packages
import matplotlib.pyplot as plt
import pandas as pd

dfp = pd.DataFrame(df)
dfp.hist(column="AGE", bins=10)
plt.show()

#### Show a bar chart with Seaborn

In [ ]:
# add seaborn to Packages
import seaborn as sns

query = """select age, count(*) occurances
from customers_fake
group by 1
order by 1"""
df = session.sql(query).to_pandas()
sns.barplot(df, x="AGE", y="OCCURANCES")